# Phase 3: is EVA-X's advantage pretraining or recipe?

Phase 2 found that EVA-X-S beats the five-model ImageNet ensemble **only when it uses its own fine-tuning recipe**
(plain BCE, layer decay, no Mixup, weight decay 0.05). Under the ImageNet models' recipe it loses. The ImageNet models
were never trained with the EVA-X recipe, so the comparison so far mixes pretraining and recipe.

This notebook trains the ImageNet backbones with the same EVA-X recipe (into `runs_v3/evaxrecipe/`, next to EVA-X-S),
adds seeds, and reruns the analysis. It never touches `runs/`, `results/`, or the existing `runs_v3/main` runs.

Upload the updated `v3` folder first (it replaces `analysis_v3.py`, `verify_run.py`, `train_v3.py`).

| Cell | What | A100 time (estimate) |
|---|---|---|
| 3 | Verify all runs, including EVA-X and the ViT rerun | 5 min, no GPU |
| 4 | **DenseNet121 with the EVA-X recipe** (decides pretraining vs recipe for the best CNN) | ~1–1.5 h |
| 5 | Analysis: EVA-X-S vs DenseNet121, same recipe | ~5 min, no GPU |
| 6 | Optional: the other four backbones with the EVA-X recipe, then the full same-recipe ensemble analysis | ~4–5 h |
| 7 | Optional: seeds 1 and 2 for EVA-X-S and DenseNet121 (EVA-X recipe) | ~3 h |

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install timm albumentations kagglehub huggingface_hub

import sys, os, glob, json, importlib
import numpy as np
hits = glob.glob('/content/drive/MyDrive/Colab Notebooks/cxr_experiment/cxr_pipeline.py') or \
       glob.glob('/content/drive/MyDrive/**/cxr_experiment/cxr_pipeline.py', recursive=True)
PROJECT = os.path.dirname(hits[0])
V3 = f'{PROJECT}/v3'
sys.path[:0] = [V3, PROJECT]
import cxr_pipeline as cx, analysis_v3, verify_run, train_v3, evax
for m in (cx, analysis_v3, verify_run, train_v3, evax): importlib.reload(m)
LOCAL_CACHE = '/content/cache'
OUT = f'{PROJECT}/results_v3b'
os.makedirs(OUT, exist_ok=True)
print('Project:', PROJECT)
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Data (copies the cache from Drive)

In [ ]:
cx.get_data(LOCAL_CACHE, f'{PROJECT}/cache')

## 3. Verify every run, including `runs_v3` (read-only). Stop if anything says FAIL.

In [ ]:
!python "{V3}/verify_run.py" --project "{PROJECT}" --cache {LOCAL_CACHE} --report "{OUT}/verify_report.json"

## 4. DenseNet121 with the EVA-X recipe (GPU)
Same data, same validation-only checkpoint selection, same recipe as EVA-X-S in `runs_v3/evaxrecipe/`.

In [ ]:
train_v3.train(PROJECT, LOCAL_CACHE, 'densenet121', recipe='evax')

## 5. Same-recipe comparison: EVA-X-S vs DenseNet121 (CPU)
Reads only `runs_v3/evaxrecipe/`. The two models differ only in architecture and pretraining.

In [ ]:
R2 = analysis_v3.analyze(f'{PROJECT}/runs_v3', LOCAL_CACHE, f'{OUT}/samerecipe_dense_vs_evax',
                         variant='evaxrecipe', n_boot=1000, bench_json=f'{PROJECT}/results_v3/bench_v3.json',
                         extra_comparisons=[('FMvsDense', 'EVA-X-S vs DenseNet121, same recipe', 'evax_s', 'densenet121')])
for c in R2['primary_comparisons']:
    print(f"{c['comparison'][:60]:60} dAUROC {c['auroc_delta']:+.4f} [{c['auroc_ci'][0]:+.4f}, {c['auroc_ci'][1]:+.4f}]  p_Holm {c['auroc_p_holm']:.3f}")

## 6. Optional: the other four backbones with the EVA-X recipe (GPU, ~4–5 h)
Answers the paper's question with the recipe held fixed: five-model ImageNet ensemble vs EVA-X-S, both EVA-X recipe.

In [ ]:
RUN_FULL_SAME_RECIPE = False
if RUN_FULL_SAME_RECIPE:
    for key in ['resnet18', 'coatnet0', 'swin_b', 'vit_b16']:
        train_v3.train(PROJECT, LOCAL_CACHE, key, recipe='evax')
    R6 = analysis_v3.analyze(f'{PROJECT}/runs_v3', LOCAL_CACHE, f'{OUT}/samerecipe_full', variant='evaxrecipe',
                             n_boot=1000, bench_json=f'{PROJECT}/results_v3/bench_v3.json',
                             extra_comparisons=[('FMvsDense', 'EVA-X-S vs DenseNet121, same recipe', 'evax_s', 'densenet121')])

## 7. Optional: training variability (GPU, ~3 h)
Seeds 1 and 2 for EVA-X-S and DenseNet121 under the EVA-X recipe. Reports the spread of test AUROC across seeds,
to compare against the size of the differences between models.

In [ ]:
RUN_SEEDS = False
if RUN_SEEDS:
    for seed in [1, 2]:
        for key in ['evax_s', 'densenet121']:
            train_v3.train(PROJECT, LOCAL_CACHE, key, recipe='evax', seed=seed, weights_dir='/content/hf')
    meta, Y, idx, _ = cx._load_split(LOCAL_CACHE)
    seeds = {}
    for key in ['evax_s', 'densenet121']:
        for v in ['evaxrecipe', 'evaxrecipe_seed1', 'evaxrecipe_seed2']:
            p = f'{PROJECT}/runs_v3/{v}/{key}/preds.npz'
            if os.path.exists(p):
                d = np.load(p)
                seeds.setdefault(key, {})[v] = float(np.nanmean(analysis_v3.auroc(Y[d['test_rows']], d['test'].astype(float))))
    for k, v in seeds.items():
        a = np.array(list(v.values()))
        print(k, v, f'mean {a.mean():.4f}  sd {a.std(ddof=1) if len(a) > 1 else float("nan"):.4f}')
    json.dump(seeds, open(f'{OUT}/seed_test_auroc.json', 'w'), indent=2)

## 8. Package (send `results_v3b.zip`, plus the `runs_v3` histories it includes)

In [ ]:
import shutil
for v in os.listdir(f'{PROJECT}/runs_v3'):
    for k in os.listdir(f'{PROJECT}/runs_v3/{v}'):
        for f in ('config.json', 'history.json'):
            src = f'{PROJECT}/runs_v3/{v}/{k}/{f}'
            if os.path.exists(src):
                os.makedirs(f'{OUT}/runs_v3/{v}/{k}', exist_ok=True)
                shutil.copy(src, f'{OUT}/runs_v3/{v}/{k}/{f}')
print(shutil.make_archive(f'{PROJECT}/results_v3b', 'zip', OUT))